# 🛠️ Notebook 3: Build a Tiny Metrics Collector

In this notebook we build a **mini Prometheus** — a tiny in-memory metrics collector with counters, gauges, and histograms. Then we plot a latency histogram with matplotlib.

The point is to understand what real metrics systems are doing under the hood: it is much simpler than it sounds.

## Learning objectives
- Implement counters, gauges, and bucketed histograms in ~30 lines.
- Compute percentiles (p50/p95/p99) from raw samples.
- Plot a latency histogram.

In [ ]:
import time, random, bisect

class Registry:
    def __init__(self):
        self._counters: dict[str, float] = {}
        self._gauges:   dict[str, float] = {}
        self._samples:  dict[str, list[float]] = {}

    # --- counters ---
    def inc(self, name, by=1.0): self._counters[name] = self._counters.get(name, 0.0) + by

    # --- gauges ---
    def set(self, name, v):      self._gauges[name] = v

    # --- histograms ---
    def observe(self, name, v):  self._samples.setdefault(name, []).append(v)

    def percentile(self, name, p):
        s = sorted(self._samples.get(name, []))
        if not s: return None
        k = max(0, min(len(s) - 1, int(round(p * (len(s) - 1)))))
        return s[k]

    def render(self):
        # Like Prometheus's text format — humans and scrapers can read it.
        lines = []
        for k, v in self._counters.items(): lines.append(f"# TYPE {k} counter\n{k} {v}")
        for k, v in self._gauges.items():   lines.append(f"# TYPE {k} gauge\n{k} {v}")
        for k in self._samples:
            lines.append(f"# TYPE {k} summary")
            for p in (0.5, 0.95, 0.99):
                lines.append(f'{k}{{quantile="{p}"}} {self.percentile(k, p):.3f}')
        return "\n".join(lines)

m = Registry()

In [ ]:
# Pretend our server handles 5000 requests with a long-tail latency distribution.
random.seed(0)
for _ in range(5000):
    latency = random.lognormvariate(4, 0.6)   # heavy right tail
    m.observe("http_latency_ms", latency)
    m.inc("http_requests_total")
    if latency > 200:
        m.inc("http_requests_slow_total")

m.set("memory_in_use_mb", 412)
print(m.render())

In [ ]:
import matplotlib.pyplot as plt

samples = m._samples["http_latency_ms"]
plt.figure(figsize=(8, 4))
plt.hist(samples, bins=60, color="steelblue", edgecolor="white")
for p, color in [(0.5, "green"), (0.95, "orange"), (0.99, "red")]:
    v = m.percentile("http_latency_ms", p)
    plt.axvline(v, color=color, linestyle="--", label=f"p{int(p*100)} = {v:.0f} ms")
plt.title("HTTP request latency")
plt.xlabel("latency (ms)")
plt.ylabel("count")
plt.legend()
plt.tight_layout()
plt.show()

## ✅ Recap

A real metrics system (Prometheus, Datadog, etc.) is doing roughly the same thing as this notebook — just with:

- bucketed histograms instead of full sample arrays (saves memory),
- a sliding time window so you see *recent* values, not lifetime,
- pull/push over the network, with labels for slicing by host, endpoint, etc.

But the core ideas — counters, gauges, percentiles — are exactly what we built here.